In [1]:
!pip install datasets==2.19.0 fsspec==2024.3.1 -q
!pip install transformers evaluate seqeval -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
s3fs 2025.3.2 requires fsspec==2025.3.2.*, but you have fsspec 2024.3.1 which is incompatible.


In [1]:
from datasets import load_dataset

dataset = load_dataset("conll2003")

print(dataset)

c:\Users\raape\anaconda3\Lib\site-packages\datasets\load.py:1486: FutureWarning: The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [2]:
#✅ STEP 3: Tokenization + Label Alignment
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [4]:
#alignment function
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(example["pos_tags"]):  # using POS here
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [5]:
#apply it
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [6]:
#STEP 4: Model Setup
from transformers import AutoModelForTokenClassification

label_list = dataset["train"].features["pos_tags"].feature.names

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label={i: l for i, l in enumerate(label_list)},
    label2id={l: i for i, l in enumerate(label_list)}
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized be

In [16]:
#STEP 5: Training Arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01
)

In [17]:
#6. Evaluation Metrics
import numpy as np
from evaluate import load

metric = load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

In [18]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [20]:
#7. trainer
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].select(range(1000)),
    eval_dataset=tokenized_dataset["validation"].select(range(300)),
    data_collator=data_collator,   # ✅ ADD THIS
    compute_metrics=compute_metrics
)

In [21]:
#train
trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=1.4045616455078125, metrics={'train_runtime': 597.2627, 'train_samples_per_second': 3.349, 'train_steps_per_second': 0.419, 'total_flos': 37315287285504.0, 'train_loss': 1.4045616455078125, 'epoch': 2.0})

In [25]:
def predict(sentence):
    tokens = sentence.split()
    inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)

    outputs = model(**inputs).logits
    predictions = outputs.argmax(dim=-1)

    word_ids = inputs.word_ids()
    result = []

    for i, word_id in enumerate(word_ids):
        if word_id is None:
            continue
        label = label_list[predictions[0][i]]
        result.append((tokens[word_id], label))

    return result

In [26]:
predictions = trainer.predict(tokenized_dataset["validation"].select(range(300)))
compute_metrics((predictions.predictions, predictions.label_ids))

{'precision': np.float64(0.8047583081570997),
 'recall': np.float64(0.7866371354743448),
 'f1': np.float64(0.7955945491879785),
 'accuracy': 0.8510455456889143}

In [27]:
predict("John works at Google in California")

[('John', 'NNP'),
 ('works', 'VBZ'),
 ('at', 'IN'),
 ('Google', 'NNP'),
 ('in', 'IN'),
 ('California', 'NNP')]

✅ Task 7: Comparison – POS Tagging vs Chunking

🔹1. Definition

   POS Tagging (Part-of-Speech Tagging):

    Assigns grammatical labels to each word in a sentence such as noun, verb, adjective, etc.

   Chunking (Shallow Parsing):

    Groups words into meaningful phrases like noun phrases (NP) or verb phrases (VP).



🔹 2. Level of Analysis

  POS Tagging: Word-level classification

  Chunking: Phrase-level grouping

🔹 3. Example

Sentence:

John works at Google in California

 Word	->    POS Tag	  ->      Chunk Tag

John    ->	 NNP	    ->       B-NP

works	->     VBZ	    ->       B-VP

at	  ->       IN	    ->           B-PP

Google	   ->  NNP	    ->       B-NP

in	   ->      IN	      ->         B-PP

California ->	 NNP	   ->        B-NP

🔹 4. Complexity

POS Tagging → Easier

Chunking → More complex (needs context understanding)

🔹 5. Purpose

POS Tagging:

Grammar analysis

Syntax understanding

Chunking:

Phrase detection

Information extraction

🔹 6. Model Behavior 

The transformer model performs well on POS tagging because it is a simpler task.

Chunking requires understanding relationships between words, making it slightly more challenging.

Proper label alignment and tokenization are crucial for both tasks.

🔹 7. Conclusion

POS tagging provides basic grammatical structure, while chunking builds on it to identify meaningful phrases. Both are essential steps in Natural Language Processing pipelines, with chunking offering deeper linguistic insights.

## 🔹 Challenges & Errors Faced

During the implementation of this task, several practical issues were encountered and resolved:

- **Dataset Loading Error:**  
  Faced issues with loading datasets like CoNLL-2003 and Universal Dependencies due to updates in the datasets library. Resolved by using a compatible version.

- **Library Compatibility Issues:**  
  Encountered version conflicts between datasets, transformers, and fsspec. Fixed by installing stable versions.

- **Kernel & Environment Issues:**  
  Faced problems with kernel selection in VS Code due to environment mismatch between WSL and Anaconda. Resolved by setting up a proper virtual environment.

- **Tokenization & Label Alignment:**  
  Difficulty in aligning labels with subword tokens. Solved using proper handling of `word_ids()` and assigning `-100` to special tokens.

- **Training Errors:**  
  Errors related to tensor shape mismatch were resolved using `DataCollatorForTokenClassification` for proper padding.

- **Trainer Compatibility Issues:**  
  Some arguments like `evaluation_strategy` and `tokenizer` were not supported due to older transformer versions. Adjusted code accordingly.

- **Evaluation Issues:**  
  Faced issues while evaluating the model due to incorrect execution order and trainer callbacks. Resolved by following proper training-evaluation sequence.

These challenges helped in gaining a deeper understanding of real-world NLP model implementation and debugging.